# Сводная статистика по портфелю

Ноутбук собирает статистику в разрезе:

**Сегмент → Зона проблемности → Тип операции (Актив / УО)**

и суммирует **Задолженность** по 8 взаимоисключающим группам с учетом **НИ, ПФН, НВВ, рестры, обеспеченности и валюты**.

Задолженность из исходного поля `Задолженность тыс. BYN` делится на 1000, поэтому итоговая сводная таблица получается в **млн BYN**.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)


## 1. Настройки

Укажите путь к исходному Excel-файлу и путь для сохранения результата.

In [ ]:
# Исходный Excel-файл
INPUT_FILE = Path('Портфель.xlsx')

# Итоговый Excel-файл
OUTPUT_FILE = Path('Статистика_портфеля.xlsx')

# 0 = первый лист Excel-файла
SHEET_NAME = 0


## 2. Вспомогательные функции

In [ ]:
def normalize_column_name(x):
    """Нормализация названия колонки для более устойчивого поиска."""
    return (
        str(x)
        .replace('\\n', ' ')
        .replace('\\r', ' ')
        .strip()
        .lower()
        .replace('ё', 'е')
    )


def find_column(df, variants):
    """Ищет колонку по одному из возможных вариантов названия."""
    normalized_columns = {
        normalize_column_name(col): col
        for col in df.columns
    }

    # Точное совпадение
    for variant in variants:
        variant_norm = normalize_column_name(variant)
        if variant_norm in normalized_columns:
            return normalized_columns[variant_norm]

    # Частичное совпадение
    for variant in variants:
        variant_norm = normalize_column_name(variant)
        for normalized_col, original_col in normalized_columns.items():
            if variant_norm in normalized_col:
                return original_col

    raise KeyError(
        f'Не найдена колонка. Искомые варианты: {variants}\\n'
        f'Колонки в файле: {list(df.columns)}'
    )


def prepare_flag(series):
    """Приводит флаг к 0/1. Поддерживает 0/1, Да/Нет, True/False."""
    s = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(',', '.', regex=False)
    )

    mapping = {
        '1': 1, '1.0': 1, 'да': 1, 'yes': 1, 'true': 1,
        '0': 0, '0.0': 0, 'нет': 0, 'no': 0, 'false': 0,
        'nan': 0, 'none': 0, '': 0,
    }

    result = s.map(mapping)
    numeric = pd.to_numeric(s, errors='coerce')
    return result.fillna(numeric)


def prepare_number(series):
    """Приводит задолженность к числу, включая значения вида '1 234,56'."""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce').fillna(0)

    s = (
        series.astype(str)
        .str.replace('\\xa0', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    return pd.to_numeric(s, errors='coerce').fillna(0)


## 3. Чтение исходного файла

In [ ]:
df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

print(f'Строк в исходном файле: {len(df):,}')
print('\\nКолонки в файле:')
display(pd.DataFrame({'Колонка': df.columns}))

display(df.head())


## 4. Поиск нужных колонок

In [ ]:
COL_SEGMENT = find_column(df, ['Сегмент'])
COL_ZONE = find_column(df, ['Зона проблемности', 'Зона'])
COL_OPERATION = find_column(df, ['Тип операции', 'Операция'])
COL_CURRENCY = find_column(df, ['Валюта', 'Код валюты', 'Валюта договора'])
COL_DEBT = find_column(df, [
    'Задолженность тыс. BYN',
    'Задолженность, тыс. BYN',
    'Задолженность'
])
COL_NI = find_column(df, ['НИ'])
COL_PFN = find_column(df, ['ПФН'])
COL_NVV = find_column(df, ['НВВ'])
COL_RESTRA = find_column(df, ['Рестра', 'Реструктуризация', 'Реструкт'])
COL_COLLATERAL = find_column(df, ['Обеспеченность'])

columns_used = pd.DataFrame({
    'Поле': [
        'Сегмент', 'Зона проблемности', 'Тип операции', 'Валюта',
        'Задолженность', 'НИ', 'ПФН', 'НВВ', 'Рестра', 'Обеспеченность'
    ],
    'Найденная колонка': [
        COL_SEGMENT, COL_ZONE, COL_OPERATION, COL_CURRENCY, COL_DEBT,
        COL_NI, COL_PFN, COL_NVV, COL_RESTRA, COL_COLLATERAL
    ]
})

display(columns_used)


## 5. Подготовка данных

In [ ]:
df['_НИ'] = prepare_flag(df[COL_NI])
df['_ПФН'] = prepare_flag(df[COL_PFN])
df['_НВВ'] = prepare_flag(df[COL_NVV])
df['_Рестра'] = prepare_flag(df[COL_RESTRA])

# Исходное поле — тыс. BYN. Делим на 1000 -> млн BYN.
df['_Задолженность'] = prepare_number(df[COL_DEBT]) / 1000

# Нормализуем тип операции.
operation_normalized = (
    df[COL_OPERATION]
    .fillna('')
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace('ё', 'е')
)

df['_Актив'] = operation_normalized.eq('актив')
df['_УО'] = operation_normalized.eq('уо')

# Нормализуем валюту. Для НВВ важен признак: валюта != BYN.
currency_normalized = (
    df[COL_CURRENCY]
    .fillna('')
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(' ', '', regex=False)
)

BYN_VALUES = {
    'BYN', '933', 'БЕЛОРУССКИЙРУБЛЬ', 'БЕЛ.РУБ.', 'БЕЛРУБ'
}

df['_Не_BYN'] = ~currency_normalized.isin(BYN_VALUES)

# Обеспеченность делим только на 2 состояния:
# 1) необеспеченный; 2) вся остальная обеспеченность.
collateral = (
    df[COL_COLLATERAL]
    .fillna('')
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace('ё', 'е')
)

df['_Необеспеченный'] = collateral.str.contains(
    r'\bне\s*обеспеч|\bнеобеспеч',
    regex=True,
    na=False
)
df['_Остальная_обеспеченность'] = ~df['_Необеспеченный']


## 6. Формирование 8 критериев

Критерии взаимоисключающие. Используется следующий приоритет:

1. **Рестра** — если `Рестра = 1`, остальные признаки не важны.
2. **ПФН** — если `ПФН = 1`, группа определяется по обеспеченности. ПФН имеет приоритет над НИ.
3. **НВВ** — только для **Активов**, только если `НИ = 0`, `ПФН = 0`, `НВВ = 1` и валюта договора **не BYN**. Обеспеченность не важна.
4. **НИ** — `НИ = 1`, `ПФН = 0`, группа определяется по обеспеченности.
5. Остаток без НИ и ПФН делится по обеспеченности. Для **УО НВВ в этих двух остаточных группах не учитывается**.

Таким образом строки с ПФН+НИ одновременно попадут в ПФН, что сохраняет прежний приоритет ПФН и не оставляет такие записи без классификации.


In [ ]:
C1 = '1. Рестра'
C2 = '2. ПФН + необеспеченный'
C3 = '3. ПФН + остальная обеспеченность'
C4 = '4. НВВ (Актив, валюта не BYN)'
C5 = '5. НИ + необеспеченный'
C6 = '6. НИ + остальная обеспеченность'
C7 = '7. Только необеспеченный'
C8 = '8. Без НИ/ПФН + остальная обеспеченность'

conditions = [
    # 1. Рестра — максимальный приоритет.
    df['_Рестра'].eq(1),

    # 2. ПФН + необеспеченный. НИ при наличии ПФН уже не влияет.
    (
        df['_Рестра'].eq(0)
        & df['_ПФН'].eq(1)
        & df['_Необеспеченный']
    ),

    # 3. ПФН + вся остальная обеспеченность.
    (
        df['_Рестра'].eq(0)
        & df['_ПФН'].eq(1)
        & df['_Остальная_обеспеченность']
    ),

    # 4. Только НВВ: нет ПФН/НИ, только Актив, валюта договора не BYN.
    # Обеспеченность значения не имеет.
    (
        df['_Рестра'].eq(0)
        & df['_ПФН'].eq(0)
        & df['_НИ'].eq(0)
        & df['_НВВ'].eq(1)
        & df['_Актив']
        & df['_Не_BYN']
    ),

    # 5. НИ без ПФН + необеспеченный.
    (
        df['_Рестра'].eq(0)
        & df['_ПФН'].eq(0)
        & df['_НИ'].eq(1)
        & df['_Необеспеченный']
    ),

    # 6. НИ без ПФН + остальная обеспеченность.
    (
        df['_Рестра'].eq(0)
        & df['_ПФН'].eq(0)
        & df['_НИ'].eq(1)
        & df['_Остальная_обеспеченность']
    ),

    # 7. Нет НИ/ПФН, необеспеченный.
    # Для УО НВВ не важен. Для Актива НВВ в валюте уже перехвачен C4.
    (
        df['_Рестра'].eq(0)
        & df['_ПФН'].eq(0)
        & df['_НИ'].eq(0)
        & df['_Необеспеченный']
    ),

    # 8. Нет НИ/ПФН и не является необеспеченным.
    # Для УО НВВ также не важен.
    (
        df['_Рестра'].eq(0)
        & df['_ПФН'].eq(0)
        & df['_НИ'].eq(0)
        & df['_Остальная_обеспеченность']
    ),
]

choices = [C1, C2, C3, C4, C5, C6, C7, C8]

df['Критерий'] = np.select(
    conditions,
    choices,
    default='НЕ РАСПРЕДЕЛЕНО'
)

print('Распределение строк по критериям:')
display(df['Критерий'].value_counts(dropna=False).to_frame('Количество строк'))

# Диагностика спорных сочетаний.
diagnostics = pd.DataFrame({
    'Проверка': [
        'Одновременно НИ=1 и ПФН=1 (относятся в ПФН)',
        'Актив: НВВ=1, но валюта BYN (НВВ-критерий не применяется)',
    ],
    'Количество строк': [
        int((df['_НИ'].eq(1) & df['_ПФН'].eq(1)).sum()),
        int((df['_Актив'] & df['_НВВ'].eq(1) & ~df['_Не_BYN']).sum()),
    ]
})

display(diagnostics)


## 7. Оставляем операции «Актив» и «УО»

In [ ]:
operation_normalized = (
    df[COL_OPERATION]
    .fillna('')
    .astype(str)
    .str.strip()
    .str.lower()
)

df_result = df[
    operation_normalized.isin(['актив', 'уо'])
].copy()

print(f'Строк после отбора Актив/УО: {len(df_result):,}')


## 8. Построение сводной таблицы

In [ ]:
criteria_order = [C1, C2, C3, C4, C5, C6, C7, C8]

summary = (
    df_result[df_result['Критерий'] != 'НЕ РАСПРЕДЕЛЕНО']
    .pivot_table(
        index=[COL_SEGMENT, COL_ZONE, COL_OPERATION],
        columns='Критерий',
        values='_Задолженность',
        aggfunc='sum',
        fill_value=0,
    )
    .reindex(columns=criteria_order, fill_value=0)
    .reset_index()
)

summary['ИТОГО'] = summary[criteria_order].sum(axis=1)

display(summary.head(20))


## 9. Контроль сумм

Этот блок помогает проверить, что задолженность не потерялась.

In [ ]:
control = pd.DataFrame({
    'Показатель': [
        'Количество исходных строк',
        'Общая задолженность исходного файла, млн BYN',
        'Задолженность Актив + УО, млн BYN',
        'Распределено по 8 критериям, млн BYN',
        'Не распределено, млн BYN',
    ],
    'Значение': [
        len(df),
        df['_Задолженность'].sum(),
        df_result['_Задолженность'].sum(),
        df_result.loc[
            df_result['Критерий'] != 'НЕ РАСПРЕДЕЛЕНО',
            '_Задолженность'
        ].sum(),
        df_result.loc[
            df_result['Критерий'] == 'НЕ РАСПРЕДЕЛЕНО',
            '_Задолженность'
        ].sum(),
    ]
})

unclassified = df_result[
    df_result['Критерий'] == 'НЕ РАСПРЕДЕЛЕНО'
].copy()

display(control)


## 10. Сохранение результата в Excel

Используется `openpyxl`, поэтому `xlsxwriter` не требуется.

Файл содержит три листа:

- **Свод** — основная таблица в млн BYN;
- **Контроль** — проверка сумм;
- **Не распределено** — строки, которые не попали ни в один критерий.


In [ ]:
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    summary.to_excel(writer, sheet_name='Свод', index=False)
    control.to_excel(writer, sheet_name='Контроль', index=False)
    unclassified.to_excel(writer, sheet_name='Не распределено', index=False)

print(f'Готово: {OUTPUT_FILE.resolve()}')
